# LangGraph with SystemMessage and HumanMessage

This notebook demonstrates how to use **SystemMessage** and **HumanMessage** in LangGraph to connect edges and nodes.

In [ ]:
# Install ipykernel if needed
# !pip install ipykernel

# Import required libraries
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import HumanMessage, SystemMessage, BaseMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
import operator

## Step 1: Define the State

The state will hold messages that flow through the graph.

In [ ]:
class GraphState(TypedDict):
    """State for our LangGraph with messages"""
    messages: Annotated[Sequence[BaseMessage], add_messages]
    next_step: str

## Step 2: Define Node Functions

Each node processes messages and can add new messages to the state.

In [ ]:
def initialize_node(state: GraphState) -> GraphState:
    """Initialize with a system message"""
    print("\n=== INITIALIZE NODE ===")
    system_msg = SystemMessage(
        content="You are a helpful assistant that processes user requests step by step."
    )
    state["messages"] = [system_msg] + list(state.get("messages", []))
    state["next_step"] = "process"
    print(f"Added SystemMessage: {system_msg.content}")
    return state


def process_node(state: GraphState) -> GraphState:
    """Process the user's message"""
    print("\n=== PROCESS NODE ===")
    messages = state.get("messages", [])
    
    # Find the last human message
    human_msg = None
    for msg in reversed(messages):
        if isinstance(msg, HumanMessage):
            human_msg = msg
            break
    
    if human_msg:
        print(f"Processing HumanMessage: {human_msg.content}")
        
        # Simulate processing and create a response
        response = AIMessage(
            content=f"I received your message: '{human_msg.content}'. Processing it now..."
        )
        state["messages"].append(response)
        state["next_step"] = "finalize"
        print(f"Added AIMessage: {response.content}")
    else:
        print("No HumanMessage found!")
        state["next_step"] = "finalize"
    
    return state


def finalize_node(state: GraphState) -> GraphState:
    """Finalize the conversation"""
    print("\n=== FINALIZE NODE ===")
    final_msg = AIMessage(
        content="Task completed successfully!"
    )
    state["messages"].append(final_msg)
    print(f"Added final AIMessage: {final_msg.content}")
    return state

## Step 3: Define Router Function

This function determines which node to visit next based on the state.

In [ ]:
def router(state: GraphState) -> str:
    """Route to the next node based on next_step"""
    next_step = state.get("next_step", "END")
    print(f"\n>>> Routing to: {next_step}")
    return next_step

## Step 4: Build the Graph

Connect nodes with edges using conditional routing.

In [ ]:
def build_message_graph():
    """Build a LangGraph with SystemMessage and HumanMessage routing"""
    
    # Create the graph
    workflow = StateGraph(GraphState)
    
    # Add nodes
    workflow.add_node("initialize", initialize_node)
    workflow.add_node("process", process_node)
    workflow.add_node("finalize", finalize_node)
    
    # Set entry point
    workflow.set_entry_point("initialize")
    
    # Add conditional edges (connecting nodes)
    workflow.add_conditional_edges(
        "initialize",
        router,
        {
            "process": "process",
            "finalize": "finalize",
            "END": END
        }
    )
    
    workflow.add_conditional_edges(
        "process",
        router,
        {
            "finalize": "finalize",
            "END": END
        }
    )
    
    # Add edge from finalize to END
    workflow.add_edge("finalize", END)
    
    # Compile the graph
    return workflow.compile()

# Build the graph
graph = build_message_graph()
print("✅ Graph built successfully!")

## Step 5: Run the Graph

Invoke the graph with a HumanMessage.

In [ ]:
# Create initial state with a HumanMessage
initial_state = {
    "messages": [
        HumanMessage(content="Hello! Can you help me understand LangGraph?")
    ],
    "next_step": "initialize"
}

# Run the graph
print("\n" + "="*60)
print("STARTING GRAPH EXECUTION")
print("="*60)

result = graph.invoke(initial_state)

print("\n" + "="*60)
print("GRAPH EXECUTION COMPLETED")
print("="*60)

## Step 6: Display Results

Show all messages in the conversation.

In [ ]:
print("\n📝 FINAL MESSAGE HISTORY:")
print("="*60)

for i, msg in enumerate(result["messages"], 1):
    msg_type = type(msg).__name__
    print(f"\n{i}. [{msg_type}]")
    print(f"   Content: {msg.content}")

print("\n" + "="*60)

## Example 2: Multiple HumanMessages

Let's try with multiple human messages in sequence.

In [ ]:
# Create a conversation with multiple messages
conversation_state = {
    "messages": [
        HumanMessage(content="What is the weather like?"),
        HumanMessage(content="Can you also tell me about RAG?")
    ],
    "next_step": "initialize"
}

print("\n" + "="*60)
print("RUNNING GRAPH WITH MULTIPLE MESSAGES")
print("="*60)

result2 = graph.invoke(conversation_state)

print("\n📝 CONVERSATION HISTORY:")
print("="*60)

for i, msg in enumerate(result2["messages"], 1):
    msg_type = type(msg).__name__
    print(f"\n{i}. [{msg_type}]")
    print(f"   {msg.content}")

## Visualize the Graph Structure

Let's visualize the graph in multiple ways.

In [ ]:
# Method 1: ASCII Representation
print("📊 GRAPH STRUCTURE (ASCII):")
print("="*60)
print("""
    START
      |
      v
  [initialize]
      |
      | (router)
      v
   [process]
      |
      | (router)
      v
   [finalize]
      |
      v
     END
""")
print("="*60)

In [ ]:
# Method 2: Get Graph Nodes and Edges
print("\n📋 GRAPH NODES AND EDGES:")
print("="*60)

try:
    graph_obj = graph.get_graph()
    
    print("\n🔵 Nodes:")
    if hasattr(graph_obj, 'nodes'):
        for node in graph_obj.nodes:
            print(f"  - {node}")
    
    print("\n🔗 Edges:")
    if hasattr(graph_obj, 'edges'):
        for edge in graph_obj.edges:
            print(f"  - {edge}")
    
    print("\n" + "="*60)
except Exception as e:
    print(f"Could not extract graph details: {e}")

In [ ]:
# Method 3: Mermaid Diagram (Text Format)
print("\n🎨 MERMAID DIAGRAM (Copy to https://mermaid.live):")
print("="*60)

mermaid_diagram = """
graph TD
    START([START]) --> initialize[Initialize Node<br/>Add SystemMessage]
    initialize --> router1{Router<br/>next_step?}
    router1 -->|process| process[Process Node<br/>Handle HumanMessage]
    process --> router2{Router<br/>next_step?}
    router2 -->|finalize| finalize[Finalize Node<br/>Add Final AIMessage]
    finalize --> END([END])
    
    style START fill:#90EE90
    style initialize fill:#87CEEB
    style process fill:#FFB6C1
    style finalize fill:#DDA0DD
    style END fill:#FF6347
    style router1 fill:#FFD700
    style router2 fill:#FFD700
"""

print(mermaid_diagram)
print("="*60)
print("\n💡 Tip: Copy the above diagram to https://mermaid.live to see it rendered!")

In [ ]:
# Method 4: Try Mermaid PNG visualization (if supported)
print("\n🖼️ ATTEMPTING PNG VISUALIZATION:")
print("="*60)

try:
    from IPython.display import Image, display
    
    # Get the graph visualization
    graph_image = graph.get_graph().draw_mermaid_png()
    display(Image(graph_image))
    print("✅ Graph visualization displayed above!")
except ImportError:
    print("⚠️ IPython.display not available (not in Jupyter environment)")
except AttributeError:
    print("⚠️ draw_mermaid_png() not available. Install: pip install pygraphviz or grandalf")
except Exception as e:
    print(f"⚠️ Could not create PNG visualization: {e}")
    print("\n📌 Use the Mermaid diagram above instead!")

## Message Flow Summary

Let's create a visual summary of how messages flow through the graph.

In [ ]:
print("\n📨 MESSAGE FLOW SUMMARY:")
print("="*60)
print("""
1. INPUT:
   └─ HumanMessage: "Your question here"

2. INITIALIZE NODE:
   ├─ Adds: SystemMessage (context setting)
   └─ Sets: next_step = "process"

3. ROUTER (after initialize):
   └─ Routes to: Process Node

4. PROCESS NODE:
   ├─ Finds: HumanMessage from messages
   ├─ Adds: AIMessage (processing response)
   └─ Sets: next_step = "finalize"

5. ROUTER (after process):
   └─ Routes to: Finalize Node

6. FINALIZE NODE:
   ├─ Adds: AIMessage (completion message)
   └─ Routes to: END

7. OUTPUT:
   └─ Final state with all messages:
      ├─ SystemMessage
      ├─ HumanMessage
      ├─ AIMessage (processing)
      └─ AIMessage (completion)
""")
print("="*60)